# Aula 06 — Teste A/B

Este notebook acompanha as notas de aula e percorre **um único experimento** do planejamento à decisão.
Os parâmetros definidos na seção 1 são os mesmos usados em todas as seções seguintes.

| Elemento | Valor |
|---|---|
| Hipótese | Exibir o frete antes da etapa de pagamento aumenta a conversão do checkout |
| Métrica principal | Taxa de conversão do checkout |
| Métrica guarda-corpo | Receita média por pedido |
| Unidade de aleatorização | Usuário |
| Conversão base | 5,0% |
| Efeito mínimo detectável | +1 ponto percentual (+20% relativo) |
| Nível de significância | 0,05 |
| Poder | 0,80 |
| Amostra resultante | 8.150 por grupo, 16.300 no total, 14 dias |

**Dependências:** `numpy`, `scipy` e `statsmodels`.
Caso `statsmodels` não esteja instalado: `pip install statsmodels`.

## 1. Especificação do experimento

Todos os parâmetros do experimento ficam em uma única célula. Qualquer alteração
aqui se propaga para o restante do notebook, o que impede que o planejamento e a
análise divirjam.

In [ ]:
import math

# --- decisões de negócio -------------------------------------------------
CONVERSAO_BASE = 0.05      # taxa de conversão atual do checkout
MDE_ABSOLUTO   = 0.01      # menor ganho que justifica a mudança: +1 ponto percentual

# --- decisões estatísticas -----------------------------------------------
ALPHA = 0.05               # probabilidade de erro tipo I
PODER = 0.80               # probabilidade de detectar o efeito, se ele existir

# --- restrição operacional ------------------------------------------------
TRAFEGO_DIARIO = 1200      # usuários elegíveis por dia

CONVERSAO_ALVO = CONVERSAO_BASE + MDE_ABSOLUTO

print(f"conversão base : {CONVERSAO_BASE:.1%}")
print(f"conversão alvo : {CONVERSAO_ALVO:.1%}")
print(f"MDE            : {MDE_ABSOLUTO * 100:.1f} p.p. "
      f"({CONVERSAO_ALVO / CONVERSAO_BASE - 1:.0%} relativo)")

## 2. Tamanho do efeito: o *h* de Cohen

Para comparar duas proporções, o tamanho de efeito padronizado é o **h de Cohen**:

$$h = 2\arcsin\left(\sqrt{p_1}\right) - 2\arcsin\left(\sqrt{p_0}\right)$$

A transformação arco-seno estabiliza a variância, que numa proporção depende do
próprio valor da proporção. Sem ela, uma diferença de 1 ponto percentual entre 5% e
6% não seria comparável à mesma diferença entre 50% e 51%.

Abaixo o valor é calculado de duas formas — pela fórmula e pelo `statsmodels` —
para deixar explícito o que a biblioteca faz.

In [ ]:
from statsmodels.stats.proportion import proportion_effectsize

def h_de_cohen(p1, p0):
    """h de Cohen para duas proporções, pela definição."""
    return 2 * math.asin(math.sqrt(p1)) - 2 * math.asin(math.sqrt(p0))

h_manual = h_de_cohen(CONVERSAO_ALVO, CONVERSAO_BASE)
h_lib = proportion_effectsize(CONVERSAO_ALVO, CONVERSAO_BASE)

print(f"pela fórmula   : h = {h_manual:.4f}")
print(f"pelo statsmodels: h = {h_lib:.4f}")

H = h_lib

## 3. Dimensionamento da amostra

Quatro parâmetros formam um sistema — α, poder, MDE e n. Fixados três, o quarto é
determinado:

$$n = \frac{\left(z_{1-\alpha/2} + z_{\text{poder}}\right)^2}{h^2}$$

O resultado é o número de observações **por grupo**. O total do experimento é 2n.

In [ ]:
from statsmodels.stats.power import NormalIndPower

n_por_grupo = NormalIndPower().solve_power(
    effect_size=H,
    alpha=ALPHA,
    power=PODER,
    ratio=1.0,                  # n_B / n_A — alocação equilibrada
    alternative="two-sided",
)

N_POR_GRUPO = math.ceil(n_por_grupo / 50) * 50     # arredondado para cima
N_TOTAL = 2 * N_POR_GRUPO

print(f"cálculo bruto : {n_por_grupo:,.0f} por grupo")
print(f"adotado       : {N_POR_GRUPO:,} por grupo")
print(f"total         : {N_TOTAL:,} observações")

### Por que `NormalIndPower`, e não `GofChisquarePower`

As duas classes existem no `statsmodels` e é comum confundi-las:

| Classe | Situação | Tamanho de efeito | O que devolve |
|---|---|---|---|
| `NormalIndPower` | duas amostras independentes | *h* de Cohen | n **por grupo** |
| `GofChisquarePower` | uma amostra contra distribuição conhecida | *w* de Cohen | n **total** |

Um teste A/B compara dois grupos, logo a classe correta é `NormalIndPower`.

A confusão não é inofensiva. Para este caso, `GofChisquarePower(n_bins=2)` devolve
**4.071**, enquanto o valor correto é **8.143 por grupo** — metade do necessário por
grupo, e um quarto do total. Um experimento dimensionado pela classe errada opera com
poder efetivo próximo de **0,51**, e não de 0,80: falha em detectar o efeito em quase
metade das vezes em que ele existe.

A célula abaixo confirma a origem do fator 2 que separa os dois valores.

In [ ]:
from statistics import NormalDist

# A variância de phi = 2*arcsen(sqrt(p)) é 1/n. Comparando DOIS grupos,
# a variância da diferença é 1/n1 + 1/n2 = 2/n na alocação equilibrada,
# e é daí que vem o fator 2 da fórmula fechada.
z_alpha = NormalDist().inv_cdf(1 - ALPHA / 2)
z_poder = NormalDist().inv_cdf(PODER)

n_sem_fator = (z_alpha + z_poder) ** 2 / H ** 2      # errado: esquece o segundo grupo
n_com_fator = 2 * (z_alpha + z_poder) ** 2 / H ** 2  # correto

print(f"sem o fator 2 : {n_sem_fator:>9,.0f}  (o que GofChisquarePower devolveria)")
print(f"com o fator 2 : {n_com_fator:>9,.0f}  (confere com NormalIndPower)")

### Sensibilidade ao MDE

Esta é a relação mais importante do planejamento, e a menos intuitiva: como *h*
entra ao quadrado no denominador, **reduzir o MDE pela metade multiplica a amostra
por aproximadamente quatro**.

In [ ]:
print(f"{'MDE relativo':>14} {'conversão alvo':>16} {'n por grupo':>14}")
print("-" * 46)

for ganho_relativo in (0.10, 0.20, 0.30, 0.40):
    alvo = CONVERSAO_BASE * (1 + ganho_relativo)
    h = proportion_effectsize(alvo, CONVERSAO_BASE)
    n = NormalIndPower().solve_power(
        effect_size=h, alpha=ALPHA, power=PODER,
        ratio=1.0, alternative="two-sided",
    )
    print(f"{ganho_relativo:>13.0%} {alvo:>16.1%} {n:>14,.0f}")

Em produtos de conversão baixa, detectar melhorias pequenas exige volume que muitas
operações não têm. Quando o tráfego disponível é insuficiente, a resposta correta é
**rever o MDE ou não executar o experimento** — nunca rodá-lo subdimensionado e
tratar o resultado como conclusivo.

## 4. Alocação desbalanceada

A divisão não precisa ser 50/50. O parâmetro `ratio` expressa n_B/n_A.

A divisão equilibrada **minimiza o total** necessário. Desbalancear é legítimo por
razões externas à estatística — limitar a exposição a uma versão de risco, por
exemplo —, e o custo dessa decisão é o aumento do total.

In [ ]:
print(f"{'divisão':>12} {'ratio':>7} {'n_A':>9} {'n_B':>9} {'total':>10}")
print("-" * 50)

for ratio in (1.0, 2.0, 3.0, 4.0):
    n_a = NormalIndPower().solve_power(
        effect_size=H, alpha=ALPHA, power=PODER,
        ratio=ratio, alternative="two-sided",
    )
    n_b = ratio * n_a
    pct_a = 100 / (1 + ratio)
    divisao = f"{pct_a:.0f}/{100 - pct_a:.0f}"
    print(f"{divisao:>12} {ratio:>7.1f} {n_a:>9,.0f} {n_b:>9,.0f} {n_a + n_b:>10,.0f}")

## 5. Duração do teste

A duração decorre do total necessário e do tráfego elegível por dia. Duas restrições
se somam ao cálculo:

- **ciclos completos**: o comportamento varia ao longo da semana, então o teste deve
  cobrir ao menos sete dias, mesmo que a amostra seja atingida antes;
- **encerramento em dia completo**: parar no meio do dia enviesa a amostra em favor
  do perfil de usuário daquele período.

In [ ]:
dias_pela_amostra = math.ceil(N_TOTAL / TRAFEGO_DIARIO)
DIAS = max(dias_pela_amostra, 7)          # nunca menos que um ciclo semanal completo

print(f"total necessário   : {N_TOTAL:,} observações")
print(f"tráfego diário     : {TRAFEGO_DIARIO:,} usuários")
print(f"dias pela amostra  : {N_TOTAL / TRAFEGO_DIARIO:.1f} -> {dias_pela_amostra}")
print(f"duração adotada    : {DIAS} dias (ciclo semanal completo)")

**O experimento está especificado.** Nenhum dado foi coletado, e toda decisão que
afeta a validade do resultado já foi tomada e registrada. É esse registro prévio que
torna o teste interpretável no final.

A partir daqui, o notebook trata dos dados coletados ao longo dos 7 dias.

## 6. Verificação prévia: divergência da razão de amostras

Antes de olhar a métrica principal, verifica-se se a divisão entre os grupos
corresponde à planejada. A **divergência da razão de amostras** (*sample ratio
mismatch*, SRM) indica defeito de instrumentação: falha no sorteio, perda assimétrica
de eventos ou filtro aplicado a apenas um grupo.

Detectada SRM, o experimento é descartado **sem** que a métrica seja consultada — a
mesma falha que desbalanceou os grupos pode tê-los tornado não comparáveis.

In [ ]:
from scipy.stats import chisquare

def verificar_srm(n_a, n_b, limiar=0.001):
    """Testa a divisão observada contra a divisão 50/50 planejada."""
    observado = [n_a, n_b]
    esperado = [sum(observado) / 2] * 2
    estatistica, p_valor = chisquare(f_obs=observado, f_exp=esperado)
    return estatistica, p_valor, p_valor < limiar

print(f"{'divisão observada':>22} {'proporção':>14} {'qui2':>9} {'p-valor':>11}  diagnóstico")
print("-" * 78)

for n_a, n_b in ((8150, 8150), (8150, 7850), (8300, 7700), (8400, 7600)):
    qui2, p, tem_srm = verificar_srm(n_a, n_b)
    prop = f"{n_a / (n_a + n_b):.1%}/{n_b / (n_a + n_b):.1%}"
    marca = "SRM - descartar" if tem_srm else "compatível com o acaso"
    print(f"{n_a:>10,} x {n_b:>8,} {prop:>14} {qui2:>9.2f} {p:>11.5f}  {marca}")

O limiar da verificação de SRM é deliberadamente **mais estrito** que o α da métrica
principal — 0,001, e não 0,05. A verificação roda em todo experimento, então um limiar
frouxo geraria alarmes falsos com frequência incômoda; e o custo de um alarme é
descartar o experimento inteiro.

Uma divisão de 51,9/48,1, que à inspeção visual parece aceitável, tem probabilidade
inferior a 0,001% de ocorrer por acaso num experimento de 16.000 unidades. Não é ruído.

## 7. O teste

Os dados coletados, organizados em tabela de contingência 2×2:

|  | Converteram | Não converteram | Total |
|---|---|---|---|
| **A (controle)** | 408 | 7.742 | 8.150 |
| **B (tratamento)** | 504 | 7.646 | 8.150 |

A hipótese nula é a de **independência** entre grupo e desfecho: pertencer a A ou a B
não altera a probabilidade de converter.

In [ ]:
import numpy as np
from scipy.stats import chi2_contingency

CONVERSOES_A, CONVERSOES_B = 408, 504

observado = np.array([
    [CONVERSOES_A, N_POR_GRUPO - CONVERSOES_A],    # A: converteram, não converteram
    [CONVERSOES_B, N_POR_GRUPO - CONVERSOES_B],    # B: converteram, não converteram
])

# SRM primeiro: a métrica só é consultada se o experimento for válido
_, p_srm, tem_srm = verificar_srm(observado[0].sum(), observado[1].sum())
assert not tem_srm, f"SRM detectada (p={p_srm:.5f}): experimento descartado"

qui2, p_valor, gl, esperado = chi2_contingency(observado, correction=False)

print(f"qui-quadrado       : {qui2:.3f}")
print(f"p-valor            : {p_valor:.4f}")
print(f"graus de liberdade : {gl}")
print("frequências esperadas sob H0:")
print(esperado)

### A correção de Yates

O parâmetro `correction` controla a **correção de continuidade de Yates**, aplicada
por padrão em tabelas 2×2. Ela subtrai 0,5 de cada desvio antes de elevar ao
quadrado, tornando o teste mais conservador.

Com `correction=False`, o qui-quadrado em tabela 2×2 é **algebricamente equivalente
ao teste z de duas proporções** — que é justamente o teste usado no dimensionamento
da seção 3. Planeja-se com o teste z e analisa-se com o qui-quadrado: são o mesmo
teste.

A escolha deve ser declarada, não omitida.

In [ ]:
for correcao in (False, True):
    q, p, _, _ = chi2_contingency(observado, correction=correcao)
    rotulo = "com Yates" if correcao else "sem Yates"
    print(f"{rotulo:>10}: qui-quadrado = {q:.3f} | p-valor = {p:.4f}")

### Hipótese bilateral e unilateral

O qui-quadrado é construído sobre desvios **ao quadrado**: detecta que observado e
esperado divergem, sem informar a direção. Em tabela 2×2 ele é, portanto,
**bilateral por construção**.

Disso decorre uma restrição frequentemente violada: **não é possível testar uma
hipótese direcional com `chi2_contingency`**. Uma hipótese da forma H₁: p_B > p_A
exige o teste z unilateral de duas proporções.

## 8. Interpretação e decisão

O teste decide entre H₀ e H₁. A decisão de produto é outra, e considera o efeito
observado, o MDE definido no planejamento e as métricas guarda-corpo.

In [ ]:
taxa_a = observado[0, 0] / observado[0].sum()
taxa_b = observado[1, 0] / observado[1].sum()
efeito_absoluto = taxa_b - taxa_a

print(f"taxa A          : {taxa_a:.2%}")
print(f"taxa B          : {taxa_b:.2%}")
print(f"efeito absoluto : {efeito_absoluto * 100:.2f} p.p.")
print(f"efeito relativo : {taxa_b / taxa_a - 1:.1%}")
print(f"MDE planejado   : {MDE_ABSOLUTO * 100:.2f} p.p.")
print()

if p_valor >= ALPHA:
    print("Falha em rejeitar H0.")
    print("Atenção: isso não é evidência de igualdade entre A e B.")
elif efeito_absoluto < MDE_ABSOLUTO:
    print("Rejeita H0, mas o efeito é inferior ao MDE.")
    print("Estatisticamente significativo e praticamente irrelevante: não implementar.")
else:
    print("Rejeita H0 e o efeito supera o MDE.")
    print("Decisão: implementar, desde que a métrica guarda-corpo esteja estável.")

Três cuidados na leitura:

- **rejeitar H₀ não mede o efeito** — o p-valor informa sobre compatibilidade com o
  acaso, não sobre magnitude. O que se compara ao MDE é o efeito observado;
- **falhar em rejeitar H₀ não é evidência de igualdade**;
- **o p-valor não é interpretável sem o poder** — e é por isso que o poder é fixado
  antes da coleta, não calculado depois.

## 9. Contraexemplo: o mesmo experimento interrompido cedo

Suponha que o teste tivesse sido encerrado após meio dia, com 400 usuários por grupo,
e que as taxas observadas fossem **ainda mais distantes** entre si do que no
experimento completo.

In [ ]:
observado_curto = np.array([
    [20, 380],    # A: 5,00%
    [25, 375],    # B: 6,25%
])

qui2_curto, p_curto, _, _ = chi2_contingency(observado_curto, correction=False)

taxa_a_curto = observado_curto[0, 0] / observado_curto[0].sum()
taxa_b_curto = observado_curto[1, 0] / observado_curto[1].sum()

print(f"taxa A          : {taxa_a_curto:.2%}")
print(f"taxa B          : {taxa_b_curto:.2%}")
print(f"efeito absoluto : {(taxa_b_curto - taxa_a_curto) * 100:.2f} p.p. "
      f"(maior que os {efeito_absoluto * 100:.2f} p.p. do experimento completo)")
print(f"qui-quadrado    : {qui2_curto:.3f}")
print(f"p-valor         : {p_curto:.4f}")
print()
print(f"n coletado por grupo : {observado_curto[0].sum()}")
print(f"n planejado por grupo: {N_POR_GRUPO:,}")

O p-valor é alto e a conclusão correta é "falha em rejeitar H₀". A conclusão
**incorreta**, e comum, seria "não há diferença entre A e B".

Com 400 observações por grupo o poder está muito abaixo de 0,80 para um efeito desta
magnitude: o teste não tinha como detectá-lo, ainda que ele existisse. O planejamento
da seção 3 já havia estabelecido que seriam necessários 4.080 por grupo.

Este resultado não informa nada sobre o produto. Informa que o experimento foi
interrompido antes da hora.

A mesma lógica explica por que **inspecionar o resultado parcial e encerrar ao ver
significância** (*peeking*) invalida o teste: sob inspeção contínua, um teste
declarado a 5% de significância apresenta taxa real de falsos positivos de 26,1%
(MILLER, 2010).